In [1]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/sw/spack-levante/cdo-2.2.2-4z4icb/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())

2.2.2


In [4]:
import os
import xarray as xr
import pandas as pd
from cdo import Cdo   # <-- missing import

data_path_pl = "/pool/data/ERA5/E5/pl/an/1H/"
day_path_sf  = "/pool/data/ERA5/E5/sf/an/1H/"
scratch_path = "/scratch/u/u301827/paris/vertical_2019/"
final_path   = "/work/uc1275/u301827/02_MSE/paris/vertical_2019/"

os.makedirs(scratch_path, exist_ok=True)
os.makedirs(final_path,   exist_ok=True)

PARIS_LON = 2.35
PARIS_LAT = 48.86

cdo = Cdo()
#cdo.debug = True

def process_era5_paris_hourly_period(start_date, end_date, var_num, var, levels="pl"):
    start_date = pd.to_datetime(start_date)
    end_date   = pd.to_datetime(end_date)

    var_scratch = os.path.join(scratch_path, var)
    var_final   = os.path.join(final_path, var)
    os.makedirs(var_scratch, exist_ok=True)
    os.makedirs(var_final,   exist_ok=True)

    out_file = os.path.join(
        var_final,
        f"{var}_{start_date:%Y%m%d}_{end_date:%Y%m%d}_paris_hourly.nc"
    )

    if levels == "pl":
        data_path = data_path_pl
    elif levels == "sf":
        data_path = day_path_sf
    else:
        raise ValueError("levels must be 'pl' or 'sf'")

    date_index = pd.date_range(start_date, end_date, freq="D")
    daily_files = []

    for i, day in enumerate(date_index, 1):
        date_str = day.strftime("%Y-%m-%d")
        var_file = (
            f"{data_path}{var_num}/"
            f"E5{levels}00_1H_{date_str}_{var_num}.grb"
        )

        print(f"[{var}] Day {i}/{len(date_index)}: {var_file}", flush=True)

        if not os.path.exists(var_file):
            print("  -> Missing, skipping.", flush=True)
            continue

        reg_file   = os.path.join(var_scratch, f"reg_{var}_{date_str}.nc")     # <-- ADD THIS
        paris_file = os.path.join(var_scratch, f"paris_{var}_{date_str}.nc")

        # Step 1: GRIB → NetCDF (keep this)
        cdo.setgridtype(
            "regular",
            input=var_file,
            output=reg_file,
            options="-f nc --eccodes"
        )

        # Step 2+3: select variable by NAME and remap to Paris
        cdo.remapnn(
            f"lon={PARIS_LON}_lat={PARIS_LAT}",
            input=f"-selname,{var} {reg_file}",
            output=paris_file
        )

        os.remove(reg_file)  # optional but recommended to save scratch space
        daily_files.append(paris_file)

    if not daily_files:
        print("No valid ERA5 files found.")
        return None

    ds = xr.open_mfdataset(daily_files, combine="by_coords")
    ds = ds.assign_coords(time=pd.to_datetime(ds.time.values))
    ds.to_netcdf(out_file)
    ds.close()

    for f in daily_files:
        os.remove(f)

    print(f"[{var}] Wrote: {out_file}", flush=True)
    return out_file


era5_vars = {
    130: {"name": "t",   "levels": "pl"},
    129: {"name": "z",   "levels": "pl"},
    135: {"name": "w",   "levels": "pl"},   # vertical (subsidence) velocity
    131: {"name": "u",   "levels": "pl"},   # zonal wind
    132: {"name": "v",   "levels": "pl"},   # meridional wind
    159: {"name": "blh", "levels": "sf"},
    167: {"name": "2t",  "levels": "sf"},
    133: {"name": "q",   "levels": "pl"},
}

start_date = "2019-07-20"
end_date   = "2019-07-27"

for var_num, meta in era5_vars.items():
    process_era5_paris_hourly_period(
        start_date=start_date,
        end_date=end_date,
        var_num=f"{var_num:03d}",
        var=meta["name"],
        levels=meta["levels"]
    )


[2t] Day 1/8: /pool/data/ERA5/E5/sf/an/1H/167/E5sf00_1H_2019-07-20_167.grb
[2t] Day 2/8: /pool/data/ERA5/E5/sf/an/1H/167/E5sf00_1H_2019-07-21_167.grb
[2t] Day 3/8: /pool/data/ERA5/E5/sf/an/1H/167/E5sf00_1H_2019-07-22_167.grb
[2t] Day 4/8: /pool/data/ERA5/E5/sf/an/1H/167/E5sf00_1H_2019-07-23_167.grb
[2t] Day 5/8: /pool/data/ERA5/E5/sf/an/1H/167/E5sf00_1H_2019-07-24_167.grb
[2t] Day 6/8: /pool/data/ERA5/E5/sf/an/1H/167/E5sf00_1H_2019-07-25_167.grb
[2t] Day 7/8: /pool/data/ERA5/E5/sf/an/1H/167/E5sf00_1H_2019-07-26_167.grb
[2t] Day 8/8: /pool/data/ERA5/E5/sf/an/1H/167/E5sf00_1H_2019-07-27_167.grb
[2t] Wrote: /work/uc1275/u301827/02_MSE/paris/vertical_2019/2t/2t_20190720_20190727_paris_hourly.nc
